# Tối ưu Báo Cáo Logistics Olist trên Google Colab
Đây là môi trường Cloud theo yêu cầu của dự án. Notebook này sẽ:
1. Kết nối với Google Drive của bạn.
2. Tự động tải dataset gốc từ Kaggle.
3. Chạy luồng ETL (Extract-Transform-Load) theo mô hình Medallion (Bronze/Silver/Gold).
4. Hiển thị (Preview) cấu trúc dữ liệu từng lớp để chứng minh sự biến đổi.
5. Phân tích trực quan bằng các biểu đồ Plotly tương tác.

## 1. Kết nối Google Drive và Cài đặt thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub pandas numpy plotly fastparquet pyarrow

## 2. ETL Component & Lưu trữ dữ liệu chuẩn Medallion Architecture

In [ ]:
import os
import kagglehub
import shutil
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/Olist_Logistics_Project'
BRONZE_DIR = os.path.join(PROJECT_DIR, 'data', 'bronze')
SILVER_DIR = os.path.join(PROJECT_DIR, 'data', 'silver')
GOLD_DIR = os.path.join(PROJECT_DIR, 'data', 'gold')

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)

print("Đang tải Data từ Kaggle...")
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
for file in os.listdir(path):
    if file.endswith('.csv'):
        shutil.copy(os.path.join(path, file), os.path.join(BRONZE_DIR, file))

print("1. Đọc dữ liệu từ Bronze...")
orders = pd.read_csv(os.path.join(BRONZE_DIR, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(BRONZE_DIR, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(BRONZE_DIR, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(BRONZE_DIR, 'olist_sellers_dataset.csv'))

print("2. Merging & Khởi tạo Feature (chuyển sang Silver)...")
df_merged = orders.merge(order_items, on='order_id', how='inner')
df_merged = df_merged.merge(customers, on='customer_id', how='inner')
df_merged = df_merged.merge(sellers, on='seller_id', how='inner')

df_delivered = df_merged[df_merged['order_status'] == 'delivered'].copy()
for col in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    df_delivered[col] = pd.to_datetime(df_delivered[col])

df_delivered.dropna(subset=['order_delivered_customer_date'], inplace=True)
df_delivered['delivery_time_days'] = (df_delivered['order_delivered_customer_date'] - df_delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
df_delivered['estimated_error_days'] = (df_delivered['order_delivered_customer_date'] - df_delivered['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)
df_delivered['is_late'] = (df_delivered['estimated_error_days'] > 0).astype(int)

silver_path = os.path.join(SILVER_DIR, 'master_logistics.parquet')
df_delivered.to_parquet(silver_path, index=False)

print("3. Tổng hợp KPIs (chuyển sang Gold)...")
state_kpi = df_delivered.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    late_orders=('is_late', 'sum'),
    avg_delivery_time=('delivery_time_days', 'mean'),
    avg_freight_value=('freight_value', 'mean')
).reset_index()

state_kpi['ontime_delivery_rate'] = ((state_kpi['total_orders'] - state_kpi['late_orders']) / state_kpi['total_orders']) * 100
gold_kpi_path = os.path.join(GOLD_DIR, 'kpi_by_state.parquet')
state_kpi.to_parquet(gold_kpi_path, index=False)
print("Hoàn tất ETL!")

## 3. Trực quan hóa cấu trúc dữ liệu qua từng Lớp (Medallion Architecture)
Để dễ dàng theo dõi, dưới đây là hiển thị 5 dòng đầu tiên của từng kho dữ liệu trước khi chúng được lưu thành file `.parquet` (Silver/Gold) và `.csv` (Bronze).

In [ ]:
from IPython.display import display, HTML
print("🟢 LỚP BRONZE (RAW DATA): Giữ nguyên định dạng gốc.\nVí dụ Bảng Orders (olist_orders_dataset.csv):")
display(orders.head(3))
print("--------------------------------------------------------------------------------------------------------------------")
print("\n🔵 LỚP SILVER (CLEAN & CONFORMED): Đã Tích hợp và Làm sạch.\nThêm 2 cột mới cực kỳ quan trọng: 'delivery_time_days' và 'is_late'.")
display(df_delivered[['order_id', 'customer_id', 'seller_id', 'delivery_time_days', 'is_late', 'freight_value']].head(3))
print("--------------------------------------------------------------------------------------------------------------------")
print("\n🟡 LỚP GOLD (AGGREGATED & KPI): Đã Tổng hợp KPIs phục vụ Phân tích & Dashboard.\nGroupby theo Bang, tính trung bình OTD%, Thời gian và Chi phí.")
display(state_kpi.head(5))

## 4. Trực quan hoá dữ liệu (Cloud Dashboard Component)

In [ ]:
import plotly.express as px
df = pd.read_parquet(gold_kpi_path)

fig_otd = px.bar(
    df.sort_values('ontime_delivery_rate', ascending=False),
    x='customer_state', y='ontime_delivery_rate',
    color='ontime_delivery_rate', color_continuous_scale='RdYlGn',
    title="1. Tỷ lệ On-Time Delivery theo Bang"
)
fig_otd.add_hline(y=df['ontime_delivery_rate'].mean(), line_dash="dash", line_color="orange")
fig_otd.show()

fig_scatter = px.scatter(
    df, x='avg_freight_value', y='avg_delivery_time',
    size='total_orders', color='ontime_delivery_rate',
    hover_name='customer_state', color_continuous_scale='RdYlGn',
    title="2. Sự tương quan giữa Phí Ship, Thời gian giao và Trọng số Đơn hàng"
)
fig_scatter.show()